In [1]:
import numpy as np

%load_ext autoreload
%autoreload 2

from gkp_optimal_control.animation import animate_wigner
from gkp_optimal_control.brachistochrone import quantum_brachistochrone_hamiltonian
from gkp_optimal_control.plotting import (
    plot_photon_number,
    plot_wigner,
    set_plot_style,
)
from gkp_optimal_control.utils import wigner_trajectory
from gkp_optimal_control.states import gkp_states_new
import jaxquantum as jqt
import jax.numpy as jnp

set_plot_style()

In [2]:
n_fock = 80
gkp_delta = 0.3
gkp_cutoff = 10
gkp_alpha = jnp.sqrt(jnp.pi / 2)
gkp_beta = jnp.sqrt(jnp.pi / 2) * 1j

w = 1.0

vac = jqt.basis(n_fock, 0)
vac_dm = vac @ vac.dag()

gkp_0, gkp_1 = gkp_states_new(n_fock, gkp_alpha, gkp_beta, gkp_delta, gkp_cutoff)

In [3]:
overlap = (vac.dag() @ gkp_0).data.squeeze()
phi = jnp.angle(overlap)
overlap_abs = jnp.clip(jnp.abs(overlap), 0.0, 1.0)
bures_angle = jnp.arccos(overlap_abs)
min_time = bures_angle / w
evo_time = min_time * 10

In [4]:
proj_2 = vac_dm * 0
bound = 30
mu = 0
alpha = np.sqrt(np.pi / 2)
beta = 1j * np.sqrt(np.pi / 2)
delta = 0.3
iden = jqt.qeye(n_fock)

for l in range(-bound, bound + 1):
    for m in range(-bound, bound + 1):
        if m % 2 != mu:
            continue
        zeta = m * alpha + l * beta
        zeta_delta = jnp.exp(-(delta**2)) * zeta
        coeff = jnp.exp(-(1j * jnp.pi * m * l) / 2) * jnp.exp(-(jnp.abs(zeta) ** 2) / 2)
        op = (zeta_delta * jqt.create(n_fock)).expm() - iden
        proj_2 += coeff * op

proj_2 = proj_2 @ vac_dm
proj_1 = proj_2.dag()
sigma_x_eff = proj_1 + proj_2
sigma_y_eff = -1j * (proj_1 - proj_2)

In [9]:
h_optimal = jnp.sin(phi) * sigma_x_eff + jnp.cos(phi) * sigma_y_eff
norm_fro = (h_optimal.dag() * h_optimal).tr()
h_optimal /= norm_fro
h_optimal = w * h_optimal

In [10]:
n_steps = 150
x_bound = 5.0
y_bound = 5.0
grid_points = 100

tlist = jnp.linspace(0.0, 10, n_steps)
states = jqt.sesolve(h_optimal, vac, tlist)

xvec, yvec, wigner_list = wigner_trajectory(states, x_bound, y_bound, grid_points=grid_points)
frames = jnp.stack([jnp.real(jnp.asarray(w)) for w in wigner_list], axis=0)

animate_wigner(
    frames,
    xvec,
    yvec,
    title="Analytic Hamiltonian Evolution",
    save_path=f"../../vids/brachistochrone/test_ham2.mp4",
    dpi=100,
)

100% |█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| [00:00<00:00, 70778.00%/s]
